# TP 1 : Collecte et prétraitement de données textuelles par web scraping

Reproduction complète du TP incluant le scraping de données d'emploi, nettoyage NLP, extraction de compétences et analyse exploratoire.

## Partie 1 : Mise en place de l'environnement

In [1]:
import subprocess
import sys

# Install required packages
packages = [
    'requests',
    'beautifulsoup4',
    'selenium',
    'pandas',
    'numpy',
    'nltk',
    'spacy',
    'wordcloud',
    'webdriver-manager',
    'tqdm'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
import json
import re
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud

# Download spaCy French model
import subprocess
import sys
try:
    import spacy
    nlp = spacy.load("fr_core_news_sm")
except:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "fr_core_news_sm"])
    import spacy
    nlp = spacy.load("fr_core_news_sm")

# Download NLTK data
import nltk
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOP_WORDS_FR = set(stopwords.words("french"))

In [ ]:
# Configure directories for local environment
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
if not BASE_DIR or BASE_DIR == '<unknown>':
    BASE_DIR = r'c:\Users\Lenovo\Desktop\Python_Ai\AAI'

DATA_DIR = os.path.join(BASE_DIR, "tp1", "data")
BRUTES_DIR = os.path.join(DATA_DIR, "brutes")
NETTOYEES_DIR = os.path.join(DATA_DIR, "nettoyees")
ANALYSEES_DIR = os.path.join(DATA_DIR, "analysees")
RAPPORTS_DIR = os.path.join(DATA_DIR, "rapports")
SCRAPING_DIR = os.path.join(BASE_DIR, "tp1", "scrapping")

# Create directories
for folder in [BRUTES_DIR, NETTOYEES_DIR, ANALYSEES_DIR, RAPPORTS_DIR, SCRAPING_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f"✓ Dossier créé: {folder}")

# Define file paths
URLS_CSV = os.path.join(BRUTES_DIR, "urls.csv")
JOBS_JSON = os.path.join(BRUTES_DIR, "jobs.json")
JOBS_CLEAN_CSV = os.path.join(NETTOYEES_DIR, "jobs_clean.csv")

print("\n✓ Configuration complétée")

## Partie 3 : Scraping des URLs des offres d'emploi

Utilisation de BeautifulSoup pour les pages statiques et Selenium pour les pages dynamiques JavaScript.

In [ ]:
# Test initial avec BeautifulSoup
url = "https://www.emploi.ma/recherche-jobs-maroc"
try:
    res = requests.get(url, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")
    
    links = []
    for a in soup.find_all("a", href=True):
        if "offre-emploi" in a["href"]:
            link = a["href"]
            if not link.startswith("http"):
                link = "https://www.emploi.ma" + link
            links.append(link)
    
    print(f"✓ {len(links)} liens trouvés")
    print("Premiers liens:")
    for link in links[:5]:
        print(f"  - {link}")
except Exception as e:
    print(f"Erreur: {e}")

In [ ]:
# Scraping avec pagination - BeautifulSoup
from urllib.parse import urljoin

all_links_bs = set()
BASE_URL = "https://www.emploi.ma"
SEARCH_URL = "https://www.emploi.ma/recherche-jobs-maroc"

for page_num in range(1, 4):  # 3 pages
    url = f"{SEARCH_URL}?page={page_num}" if page_num > 1 else SEARCH_URL
    
    try:
        print(f"Scraping page {page_num}...")
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")
        
        for a in soup.find_all("a", href=True):
            if "offre-emploi" in a["href"]:
                link = a["href"]
                if not link.startswith("http"):
                    link = urljoin(BASE_URL, link)
                all_links_bs.add(link)
        
        # Politeness delay
        time.sleep(random.uniform(2, 4))
        print(f"  ✓ Page {page_num} complétée ({len(all_links_bs)} liens au total)")
    except Exception as e:
        print(f"  ✗ Erreur page {page_num}: {e}")
        continue

print(f"\n✓ Total liens collectés: {len(all_links_bs)}")

In [ ]:
# Scraping avec Selenium pour pages dynamiques (optionnel - commenté pour éviter les complications)
# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from webdriver_manager.chrome import ChromeDriverManager

# all_links_selenium = set()
# 
# try:
#     chrome_options = Options()
#     chrome_options.add_argument("--disable-gpu")
#     chrome_options.add_argument("--no-sandbox")
#     
#     service = Service(ChromeDriverManager().install())
#     driver = webdriver.Chrome(service=service, options=chrome_options)
#     
#     for page in range(1, 3):
#         url = f"{SEARCH_URL}?page={page}" if page > 1 else SEARCH_URL
#         print(f"Selenium - Page {page}...")
#         driver.get(url)
#         time.sleep(5)
#         
#         elements = driver.find_elements(By.TAG_NAME, "a")
#         for el in elements:
#             link = el.get_attribute("href")
#             if link and "offre-emploi" in link:
#                 all_links_selenium.add(link)
#         
#         time.sleep(random.uniform(2, 3))
#     
#     driver.quit()
#     print(f"✓ Selenium: {len(all_links_selenium)} liens collectés")
# except Exception as e:
#     print(f"Note Selenium: {e}")

# Utiliser les liens collectés
all_job_urls = list(all_links_bs)
print(f"\n✓ Total liens pour le scraping: {len(all_job_urls)}")

In [ ]:
# Sauvegarder les URLs en CSV
df_urls = pd.DataFrame({"url": all_job_urls})
df_urls.to_csv(URLS_CSV, index=False, encoding="utf-8")
print(f"✓ URLs sauvegardées: {URLS_CSV}")
print(f"  Nombre d'URLs: {len(df_urls)}")

## Partie 4 : Scraping des détails de chaque offre

In [ ]:
def fetch_offer_details(url):
    """
    Scrape les détails d'une offre d'emploi
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extraction des éléments
        title = ""
        company = ""
        description = ""
        location = ""
        contract_type = ""

        # Chercher le titre
        try:
            node = soup.select_one("h1")
            if node:
                title = node.get_text(strip=True)
        except:
            pass

        # Chercher la description
        try:
            paragraphs = soup.select("div p, article p, main p")
            desc = [p.get_text(" ", strip=True) for p in paragraphs if p.get_text(strip=True)]
            description = " ".join(desc).strip()
        except:
            pass

        # Chercher l'entreprise
        try:
            for selector in [".company", ".employer", "[data-company]"]:
                node = soup.select_one(selector)
                if node:
                    company = node.get_text(" ", strip=True)
                    break
        except:
            pass

        # Chercher la localisation
        try:
            for selector in [".location", ".ville", "[data-location]"]:
                node = soup.select_one(selector)
                if node:
                    location = node.get_text(" ", strip=True)
                    break
        except:
            pass

        # Chercher le type de contrat
        try:
            for selector in [".contract", ".contract-type", ".type", "[data-contract]"]:
                node = soup.select_one(selector)
                if node:
                    contract_type = node.get_text(" ", strip=True)
                    break
        except:
            pass
        
        return {
            "url": url,
            "title": title or "",
            "company": company or "",
            "description": description or "",
            "location": location or "",
            "contract_type": contract_type or "",
            "error": None,
            "missing_fields": [
                key for key in ["title", "company", "description", "location", "contract_type"]
                if not locals().get(key)
            ]
        }
    
    except requests.RequestException as e:
        return {
            "url": url,
            "title": "",
            "company": "",
            "description": "",
            "location": "",
            "contract_type": "",
            "error": str(e),
            "missing_fields": ["title", "company", "description", "location", "contract_type"],
        }
    except Exception as e:
        return {
            "url": url,
            "title": "",
            "company": "",
            "description": "",
            "location": "",
            "contract_type": "",
            "error": str(e),
            "missing_fields": ["title", "company", "description", "location", "contract_type"],
        }

print("✓ Fonction fetch_offer_details prête")

In [ ]:
# Scraper les détails des offres (limité à 10 pour démo)
offers_details = []
urls_to_scrape = all_job_urls[:10]  # Limiter pour la démo

print(f"Scraping {len(urls_to_scrape)} offres...\n")

for i, url in enumerate(tqdm(urls_to_scrape, desc="Scraping détails"), 1):
    details = fetch_offer_details(url)
    offers_details.append(details)
    
    # Politeness delay
    time.sleep(random.uniform(1, 2))

print(f"\n✓ {len(offers_details)} offres scrappées")
print("\nExemples:")
for offer in offers_details[:2]:
    print(f"  - {offer['title'][:50] if offer['title'] else 'N/A'}...")

In [ ]:
# Sauvegarder les offres en JSON
with open(JOBS_JSON, "w", encoding="utf-8") as f:
    json.dump(offers_details, f, ensure_ascii=False, indent=2)

print(f"✓ Offres sauvegardées: {JOBS_JSON}")

# Afficher les statistiques
df_offers = pd.DataFrame(offers_details)
print(f"\nStatistiques:")
print(f"  - Offres avec titre: {(df_offers['title'] != '').sum()}/{len(df_offers)}")
print(f"  - Offres avec description: {(df_offers['description'] != '').sum()}/{len(df_offers)}")
print(f"  - Offres avec erreurs: {(df_offers['error'] != None).sum()}/{len(df_offers)}")

## Partie 5 : Nettoyage et prétraitement des données textuelles

In [ ]:
import html

def nettoyer_texte(texte):
    """Nettoie le texte brut"""
    if not texte:
        return ""
    
    # Unescape HTML entities
    texte = html.unescape(texte)
    
    # Supprimer les tags HTML
    texte = re.sub(r"<[^>]+>", " ", texte)
    
    # Supprimer caractères spéciaux (garder accents)
    texte = re.sub(
        r"[^\w\s.,;:!?'\"\(\)\-àâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ]",
        " ",
        texte,
    )
    
    # Normaliser les retours à la ligne
    texte = re.sub(r"\n{3,}", "\n\n", texte)
    
    # Normaliser les espaces
    texte = re.sub(r" {2,}", " ", texte)
    
    return texte.strip()

def tokeniser(texte):
    """Tokenise le texte"""
    doc = nlp(texte)
    tokens = [token.text.lower() for token in doc]
    # Garder seulement les tokens avec des lettres
    tokens = [
        t for t in tokens
        if re.search(r"[a-zA-ZàâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ]", t)
    ]
    return tokens

def supprimer_stop_words(tokens):
    """Supprime les mots vides"""
    return [t for t in tokens if t.lower() not in STOP_WORDS_FR]

def lemmatiser(tokens):
    """Lemmatise les tokens"""
    doc = nlp(" ".join(tokens))
    return [token.lemma_.lower() for token in doc if token.lemma_.strip()]

def pipeline_nettoyage(texte):
    """Pipeline complet de nettoyage"""
    texte_propre = nettoyer_texte(texte)
    tokens = tokeniser(texte_propre)
    tokens = supprimer_stop_words(tokens)
    lemmes = lemmatiser(tokens)
    
    return {
        "description_nettoyee": texte_propre,
        "tokens": lemmes,
        "tokens_str": " ".join(lemmes),
    }

print("✓ Fonctions de nettoyage prêtes")

In [ ]:
# Appliquer le pipeline de nettoyage
print("Nettoyage NLP des données...\n")

rows = []
for offre in tqdm(offers_details, desc="Nettoyage NLP", unit="offre"):
    description_brute = offre.get("description") or ""
    
    if description_brute:
        resultat = pipeline_nettoyage(description_brute)
    else:
        resultat = {
            "description_nettoyee": "",
            "tokens": [],
            "tokens_str": "",
        }
    
    rows.append({
        "url": offre.get("url"),
        "title": offre.get("title"),
        "company": offre.get("company"),
        "location": offre.get("location"),
        "contract_type": offre.get("contract_type"),
        "description_brute": description_brute,
        "description_nettoyee": resultat["description_nettoyee"],
        "tokens": resultat["tokens_str"],
    })

# Créer DataFrame et sauvegarder
df_nettoyees = pd.DataFrame(rows)
df_nettoyees.to_csv(JOBS_CLEAN_CSV, index=False, encoding="utf-8")

print(f"\n✓ Données nettoyées sauvegardées: {JOBS_CLEAN_CSV}")
print(f"  Nombre d'offres: {len(df_nettoyees)}")
print("\nAperçu:")
print(df_nettoyees[["title", "description_nettoyee"]].head(2))

## Partie 6 : Analyse exploratoire et extraction de compétences

In [ ]:
# Extraction de compétences
skills_list = [
    "python", "java", "c++", "c#", "sql", "excel",
    "javascript", "typescript", "html", "css", "react", "angular", "vue",
    "django", "flask", "node", "nodejs", "spring",
    "linux", "docker", "kubernetes", "git",
    "word", "powerpoint", "aws", "azure", "gcp",
    "pytorch", "tensorflow", "keras", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn"
]

def extract_skills(text):
    """Extrait les compétences d'un texte"""
    found = []
    if not text:
        return found
    
    text = text.lower()
    for skill in skills_list:
        if skill in text:
            found.append(skill)
    
    return list(set(found))  # Supprimer les doublons

# Ajouter colonne skills au DataFrame
df_nettoyees["skills"] = df_nettoyees["description_brute"].apply(extract_skills)

print("✓ Extraction de compétences complétée")
print(f"\nExemples:")
for idx, row in df_nettoyees.head(3).iterrows():
    skills = row['skills']
    if skills:
        print(f"  - {row['title'][:40]}: {', '.join(skills)}")
    else:
        print(f"  - {row['title'][:40]}: Aucune compétence détectée")

In [ ]:
# Analyse des compétences
print("=== ANALYSE DES COMPÉTENCES ===\n")

# Analyser toutes les compétences
all_skills = []
for skills in df_nettoyees["skills"]:
    all_skills.extend(skills)

if all_skills:
    top_skills = Counter(all_skills).most_common(15)
    print(f"Top 15 compétences demandées:")
    for skill, count in top_skills:
        print(f"  {skill}: {count} mentions")
else:
    print("Aucune compétence détectée dans les descriptions")

# Statistiques générales
print(f"\n=== STATISTIQUES GÉNÉRALES ===")
print(f"Nombre total d'offres: {len(df_nettoyees)}")
print(f"Offres avec titre: {(df_nettoyees['title'] != '').sum()}")
print(f"Offres avec description: {(df_nettoyees['description_nettoyee'] != '').sum()}")

# Compagnies
print(f"\n=== TOP COMPAGNIES ===")
companies = df_nettoyees[df_nettoyees['company'] != '']['company'].value_counts()
if len(companies) > 0:
    for company, count in companies.head(5).items():
        print(f"  {company}: {count} offres")
else:
    print("  Aucune information de compagnie disponible")

# Localités
print(f"\n=== TOP LOCALITÉS ===")
locations = df_nettoyees[df_nettoyees['location'] != '']['location'].value_counts()
if len(locations) > 0:
    for location, count in locations.head(5).items():
        print(f"  {location}: {count} offres")
else:
    print("  Aucune information de localité disponible")

In [ ]:
# Visualisations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Top compétences
if all_skills:
    skills, counts = zip(*top_skills[:10])
    axes[0, 0].bar(skills, counts, color='steelblue')
    axes[0, 0].set_title('Top 10 Compétences Demandées')
    axes[0, 0].set_xlabel('Compétence')
    axes[0, 0].set_ylabel('Nombre de mentions')
    axes[0, 0].tick_params(axis='x', rotation=45)
else:
    axes[0, 0].text(0.5, 0.5, 'Aucune compétence détectée', ha='center')
    axes[0, 0].set_title('Top Compétences')

# 2. Distribution par compagnie
companies = df_nettoyees[df_nettoyees['company'] != '']['company'].value_counts()
if len(companies) > 0:
    companies.head(5).plot(kind='bar', ax=axes[0, 1], color='coral')
    axes[0, 1].set_title('Top 5 Compagnies')
    axes[0, 1].set_xlabel('Compagnie')
    axes[0, 1].set_ylabel('Nombre d\'offres')
else:
    axes[0, 1].text(0.5, 0.5, 'Pas de données de compagnie', ha='center')
    axes[0, 1].set_title('Distribution par Compagnie')

# 3. Distribution par localité
locations = df_nettoyees[df_nettoyees['location'] != '']['location'].value_counts()
if len(locations) > 0:
    locations.head(5).plot(kind='bar', ax=axes[1, 0], color='lightgreen')
    axes[1, 0].set_title('Top 5 Localités')
    axes[1, 0].set_xlabel('Localité')
    axes[1, 0].set_ylabel('Nombre d\'offres')
else:
    axes[1, 0].text(0.5, 0.5, 'Pas de données de localité', ha='center')
    axes[1, 0].set_title('Distribution par Localité')

# 4. Longueur des descriptions
df_nettoyees['desc_length'] = df_nettoyees['description_nettoyee'].str.len()
axes[1, 1].hist(df_nettoyees['desc_length'], bins=20, color='purple', alpha=0.7)
axes[1, 1].set_title('Distribution de la Longueur des Descriptions')
axes[1, 1].set_xlabel('Nombre de caractères')
axes[1, 1].set_ylabel('Nombre d\'offres')

plt.tight_layout()
plt.savefig(os.path.join(ANALYSEES_DIR, 'analyse_offres.png'), dpi=100, bbox_inches='tight')
plt.show()

print("✓ Visualisations créées")

In [ ]:
# Word Cloud des descriptions
text = " ".join(df_nettoyees["description_nettoyee"].dropna())

if text.strip():
    wordcloud = WordCloud(width=1200, height=600, background_color='white', 
                         max_words=100, colormap='viridis').generate(text)
    
    plt.figure(figsize=(15, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title("Nuage de Mots - Descriptions des Offres", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSEES_DIR, 'wordcloud.png'), dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✓ Word Cloud généré")
else:
    print("Pas de texte disponible pour générer le word cloud")

In [ ]:
# Résumé final et fichiers générés
print("="*60)
print("✓ TP 1 : COLLECTE ET PRÉTRAITEMENT TERMINÉ")
print("="*60)

print("\n📊 FICHIERS GÉNÉRÉS:")
print(f"\n1. URLs collectées:")
print(f"   - {URLS_CSV}")
print(f"   - {len(df_urls)} URLs sauvegardées")

print(f"\n2. Offres brutes (JSON):")
print(f"   - {JOBS_JSON}")
print(f"   - {len(offers_details)} offres scrappées")

print(f"\n3. Données nettoyées (CSV):")
print(f"   - {JOBS_CLEAN_CSV}")
print(f"   - {len(df_nettoyees)} offres nettoyées")

print(f"\n4. Visualisations:")
print(f"   - {os.path.join(ANALYSEES_DIR, 'analyse_offres.png')}")
print(f"   - {os.path.join(ANALYSEES_DIR, 'wordcloud.png')}")

print(f"\n📈 STATISTIQUES FINALES:")
print(f"   - Offres avec titre: {(df_nettoyees['title'] != '').sum()}/{len(df_nettoyees)}")
print(f"   - Offres avec description: {(df_nettoyees['description_nettoyee'] != '').sum()}/{len(df_nettoyees)}")
print(f"   - Compétences uniques détectées: {len(set(all_skills))}")
if all_skills:
    print(f"   - Compétence la plus demandée: {all_skills[0] if all_skills else 'N/A'}")

print("\n✓ Tous les fichiers sont prêts pour analyse ultérieure!")
print("="*60)